# Agentic AI Demo - Project Scaffolding Agent with Gemini


An **AI-powered project scaffolding agent** built with **Google Gemini 2.5 Flash**. Give it a project style and a topic, and it autonomously generates a complete folder structure with real, functional starter code - then executes the actions to scaffold everything on disk.

### What makes this Agentic AI?
Unlike a simple chatbot that just *returns text*, this agent **takes actions in the real world**:
- Tool use - The LLM doesnt just suggest code; it generates structured function calls (`create_folder`, `create_file`, `write_to_file`) that are executed on your filesystem

- Autonomy - Given only a style and topic, the agent independently decides the folder structure, file names, dependencies, and content - no human in the loop

- Action execution via `eval()` - The AI's response is parsed and each action is directly executed at runtime, not routed through hardcoded if/else or switch/case logic - the AI *decides* what functions to call and with what arguments

- Structured output - The agent is instructed to respond in a strict JSON schema, making its outputs machine-readable and executable

> **TL;DR:** A regular AI *tells* you what to do. An agentic AI *does* it.

</br>

> **Security** - Actions are executed via `eval()` with a **sandboxed `safe_globals`** dict that exposes only `create_folder`, `create_file`, and `write_to_file` - no access to `os`, `sys`, `subprocess`, builtins, or any other dangerous functions.

## What makes this not TRULY Agentic?
This is currently more like a **Structured LLM output with automated execution**. This is not a true Agentic AI workflow but a building block. A true Agentic AI will have a loop that calls the LLM, sees the result of each action and then decides what to do next. 

### How it works
1. Pick a **project style** (e.g. `rest-api`, `cli-tool`, `fullstack`, `game`, etc.)
2. Describe a **topic** (e.g. "a bookstore inventory management system")
3. Gemini responds with structured JSON containing `create_folder`, `create_file`, and `write_to_file` actions
4. The agent parses the response and **executes each action via `eval()`** - no hardcoded dispatch, pure agentic execution

### Supported Styles
`rest-api` · `cli-tool` · `web-app` · `fullstack` · `data-science` · `ml-pipeline` · `python-package` · `chrome-extension` · `discord-bot` · `mobile-app` · `microservice` · `static-site` · `game` · `vscode-extension` · `terraform`

In [ ]:
!pip install -q -U google-genai python-dotenv

import os
from dotenv import load_dotenv
from google import genai
import json

load_dotenv()  # reads API key from .env file

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def create_folder(folder_path): os.makedirs(folder_path, exist_ok=True)
def create_file(file_path): open(file_path, "a").close()
def write_to_file(file_path, file_content): open(file_path, "w").write(file_content)

In [33]:
# Supported project styles
PROJECT_STYLES = {
    "rest-api":          "REST API backend (Flask/FastAPI) with routes, models, middleware",
    "cli-tool":          "Command-line tool with argument parsing and subcommands",
    "web-app":           "Frontend web app (HTML/CSS/JS or React/Vue)",
    "fullstack":         "Full-stack app with frontend + backend + database",
    "data-science":      "Data science project with notebooks, data, and scripts",
    "ml-pipeline":       "Machine learning pipeline with training, evaluation, and inference",
    "python-package":    "Publishable Python package with setup.py, tests, and docs",
    "chrome-extension":  "Chrome browser extension with manifest, popup, and background scripts",
    "discord-bot":       "Discord bot with commands, events, and cogs",
    "mobile-app":        "Mobile app (React Native / Flutter) with screens and navigation",
    "microservice":      "Microservice with Docker, health checks, and message queue support",
    "static-site":       "Static site / blog with templates and content",
    "game":              "Simple game project (Pygame / JS Canvas) with assets and scenes",
    "vscode-extension":  "VS Code extension with commands, views, and settings",
    "terraform":         "Infrastructure-as-Code project with Terraform modules and envs",
}

print("Supported project styles:\n")
for key, desc in PROJECT_STYLES.items():
    print(f"  • {key:20s} → {desc}")

Supported project styles:

  • rest-api             → REST API backend (Flask/FastAPI) with routes, models, middleware
  • cli-tool             → Command-line tool with argument parsing and subcommands
  • web-app              → Frontend web app (HTML/CSS/JS or React/Vue)
  • fullstack            → Full-stack app with frontend + backend + database
  • data-science         → Data science project with notebooks, data, and scripts
  • ml-pipeline          → Machine learning pipeline with training, evaluation, and inference
  • python-package       → Publishable Python package with setup.py, tests, and docs
  • chrome-extension     → Chrome browser extension with manifest, popup, and background scripts
  • discord-bot          → Discord bot with commands, events, and cogs
  • mobile-app           → Mobile app (React Native / Flutter) with screens and navigation
  • microservice         → Microservice with Docker, health checks, and message queue support
  • static-site          → Static si

In [19]:
SYSTEM_PROMPT = """You are an expert software architect agent. Given a project style and topic,
you generate a complete project folder structure with file contents.

You have access to only the below functions to scaffold the project:
create_folder(path)
create_file(path)
write_to_file(path, content) 

RESPOND ONLY WITH VALID JSON in this exact format:
{
    "project_name": "my-project",
    "description": "Brief project description",
    "structure": [
        {
            "type": "folder",
            "path": "my-project",
            "action": [
                "create_folder('my-project')"
            ]
        },
        {
            "type": "file",
            "path": "my-project/requirements.txt",
            "content": '''flask==3.0\\nrequests==2.31''',
            "action": [
                "create_file('my-project/requirements.txt')",
                "write_to_file('my-project/requirements.txt', '''flask==3.0\\nrequests==2.31''')"
            ]
        },
        {
            "type": "folder",
            "path": "my-project/src",
            "action": [
                "create_folder('my-project/src')"
            ]
        },
        {
            "type": "file",
            "path": "my-project/src/main.py",
            "content": '''# main entry point\\ndef main():\\n    print('hello')''',
            "action": [
                "create_file('my-project/src/main.py')",
                "write_to_file('my-project/src/main.py', '''# main entry point\\ndef main():\\n    print('hello')''')"
            ]
        }
    ]
}

Rules:
- Ensure every structure has an action that uses only the provided functions to create/write files and folders
- CRITICAL: In content and write_to_file actions, ALWAYS wrap the content argument in TRIPLE QUOTES like '''content here'''
- This allows the content to safely contain single quotes, double quotes, and newlines
- Include ALL necessary config files (.gitignore, Dockerfile, package.json, etc.) relevant to the project style
- Write REAL, functional starter code - not just placeholders
- Include a README with setup instructions
- Include a requirements.txt or package.json with actual dependencies
- Keep files concise but functional
- Use best practices for the given project style
"""

In [20]:
# Agent that generates and scaffolds a full project from a style + topic.
def get_project_structure(style: str, topic: str):
    if style not in PROJECT_STYLES:
        print(f"Unknown style '{style}'. Choose from: {', '.join(PROJECT_STYLES.keys())}")
        return

    user_prompt = f"""Create a **{style}** project about: "{topic}"
            Style description: {PROJECT_STYLES[style]}
            Generate the full folder structure and file contents."""

    print(f"Generating {style} project: \"{topic}\"...")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_prompt,
        config={
            "system_instruction": SYSTEM_PROMPT,
            "temperature": 0.7,
        }
    )

    return response.text

In [ ]:
# Test the Agent with a sample project
response = get_project_structure("rest-api", "a bookstore inventory management system")

Generating rest-api project: "a bookstore inventory management system"...


In [6]:
# ── More examples (uncomment any to try) ──────────────────────────────

# generate_project("cli-tool",          "a file duplicate finder")
# generate_project("fullstack",         "a todo app with authentication")
# generate_project("ml-pipeline",       "sentiment analysis on product reviews")
# generate_project("discord-bot",       "a music trivia quiz bot")
# generate_project("chrome-extension",  "a website reading time estimator")
# generate_project("microservice",      "a URL shortener service")
# generate_project("data-science",      "analyzing global CO2 emissions")
# generate_project("python-package",    "a markdown-to-HTML converter library")
# generate_project("terraform",         "AWS ECS deployment with ALB")
# generate_project("game",              "a snake game with power-ups")
# generate_project("vscode-extension",  "a code snippet manager")
# generate_project("mobile-app",        "a habit tracker with streaks")
# generate_project("static-site",       "a personal developer portfolio")

In [28]:
import ast
# clean and parse json response
text = response.strip()
if text.startswith("```"):
    text = text.split("\n", 1)[1]
    text = text.rsplit("```", 1)[0]

try:
    project_instructions = json.loads(text, strict=False)
except json.JSONDecodeError:
    project_instructions = ast.literal_eval(text)

# cd to output directory
OUTPUT_DIR = os.environ["OUTPUT_DIR"]
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(OUTPUT_DIR)
print(f"Output dir: {os.getcwd()}\n")

# only expose the allowed functions to eval
safe_globals = {
    "create_folder": create_folder,
    "create_file": create_file,
    "write_to_file": write_to_file,
}

# execute each action
for item in project_instructions["structure"]:
    print(f" -- Creating: {item['type']:6s} {item['path']} ------------------")
    for action in item["action"]:
        if action.startswith("write_to_file"):
            print(f"\t .. Executing: write_to_file")
        else: 
            print(f"\t .. Executing: {action}")
        eval(action, safe_globals)

print(f" Done! Project scaffolded at: {os.path.join(OUTPUT_DIR, project_instructions['project_name'])}")

Output dir: /Users/arantarokade/Documents/speedrun-gen-ai/output

 -- Creating: folder bookstore-api ------------------
	 .. Executing: create_folder('bookstore-api')
 -- Creating: file   bookstore-api/README.md ------------------
	 .. Executing: create_file('bookstore-api/README.md')
	 .. Executing: write_to_file
 -- Creating: file   bookstore-api/requirements.txt ------------------
	 .. Executing: create_file('bookstore-api/requirements.txt')
	 .. Executing: write_to_file
 -- Creating: file   bookstore-api/.gitignore ------------------
	 .. Executing: create_file('bookstore-api/.gitignore')
	 .. Executing: write_to_file
 -- Creating: file   bookstore-api/Dockerfile ------------------
	 .. Executing: create_file('bookstore-api/Dockerfile')
	 .. Executing: write_to_file
 -- Creating: file   bookstore-api/config.py ------------------
	 .. Executing: create_file('bookstore-api/config.py')
	 .. Executing: write_to_file
 -- Creating: file   bookstore-api/run.py ------------------
	 .. Exec